# Transmon-controlled cavity–cavity SWAP, using only the **ge** transmon transition

Hardware assumed (all gates **ideal**):

* cavity–cavity beamsplitter, $g_{bs}\,(a_1^\dagger a_2 e^{i\varphi}+\mathrm{h.c.})$
* transmon–cavity ge exchange on either cavity, **or both simultaneously**
* no static $\chi$ — the transmon is idle unless a pump is on

---

## The identity everything rests on

Define the supermodes $a_\pm=(a_1\pm a_2)/\sqrt2$, $n_\pm = a_\pm^\dagger a_\pm$.

$$\mathrm{SWAP}_{12}\;=\;e^{\,i\pi n_-}\qquad\textbf{exactly.}$$

A SWAP *is* a $\pi$ phase per photon in the antisymmetric supermode: $a_+$ is untouched,
$a_-\to -a_-$, which is exactly $a_1\leftrightarrow a_2$.

Driving **both** transmon–cavity ge exchanges at once, with **equal amplitude** and
**relative phase $\pi$**, is *identically* a single-mode Jaynes–Cummings coupling to
$a_-$ with strength $G=\sqrt2\,g$ — and $a_+$ stays decoupled to all orders, not just
perturbatively. So

$$\textbf{transmon-controlled SWAP}\;=\;\textbf{transmon-controlled parity of }a_-$$

The three-body problem collapses to a **single-mode** conditional-phase problem whose
blocks are $2\times2$. No beamsplitter sandwich, no dispersive approximation, no $f$ level.

## The gate, concretely

### Primary drive

Two pumps on **at the same time**, each activating the ge exchange between the transmon
and one cavity — the same pumps you already use for a transmon–cavity beamsplitter, just
applied to both cavities together:

$$H_j=\underbrace{\Delta_j|e\rangle\langle e|}_{\text{common detuning}}
+\;g\Big(e^{i\phi_j}\,a_1^\dagger|g\rangle\langle e|+\mathrm{h.c.}\Big)
+\;g\Big(e^{i(\phi_j+\pi)}\,a_2^\dagger|g\rangle\langle e|+\mathrm{h.c.}\Big)$$

Three requirements, all set by the pumps:

| requirement | why |
|---|---|
| equal amplitudes $g_1=g_2=g$ | makes the bright mode exactly $a_-$ |
| relative phase exactly $\pi$ | picks $a_-$ (rather than $a_+$) as the bright mode |
| equal detunings $\Delta_1=\Delta_2=\Delta_j$ | keeps $a_+$ decoupled (same condition an ordinary cavity–cavity BS already needs) |

Under those, $H_j$ is *identically*

$$H_j=\Delta_j|e\rangle\langle e|+G\big(e^{i\phi_j}a_-^\dagger|g\rangle\langle e|+\mathrm{h.c.}\big),
\qquad G=\sqrt2\,g$$

Segment $j$ is fully specified by three numbers: **pulse area** $x_j=G\,t_j$, **drive
phase** $\phi_j$, **detuning** $r_j=\Delta_j/G$. The $f$ level is never used.

### Control qubit

The **ordinary transmon g–e qubit**. There is no spectator level: both control states
live in the *same* driven block $\{|n,g\rangle,\;|n\!-\!1,e\rangle\}$. That turns out to be
an advantage, not a problem:

* the ge exchange conserves $n+n_q$, so the whole propagator is block-diagonal in $2\times2$ blocks;
* demanding $|\langle n,g|U|n,g\rangle|=1$ **forces each block to be diagonal**, i.e. no
  residual g↔e mixing;
* once a block is diagonal, unitarity fixes the $e$ phase from the $g$ phase:
  $U^{ee}_m=e^{2i\gamma}/U^{gg}_m$ with $\gamma=-\sum_j\Delta_jt_j/2$ (independent of $n$).

So we only ever have to shape **one** phase profile. Target the "quarter parity"

$$\langle n,g|U|n,g\rangle=e^{\pm i\pi n/2}\qquad(\theta=\pm\pi/2)$$

and the $e$ branch automatically comes out as $e^{\mp i\pi n/2}$. (Either sign works;
it only decides which branch swaps, and the final beamsplitter absorbs the difference.) The relative phase between
the two control states is then $e^{i\pi n}$ — the conditional parity — and

$$U \;=\; \underbrace{e^{\mp i\pi n_-/2}}_{\text{unconditional 50:50 BS}}\;\times\;\text{controlled-SWAP}$$

### Full sequence

1. $M$ segments of the simultaneous, $\pi$-out-of-phase double ge drive, with the
   optimised $(x_j,\phi_j,r_j)$ below;
2. **one** unconditional cavity–cavity beamsplitter $e^{\pm i\pi n_-/2}$ (50:50 BS plus a
   virtual Z on each cavity) — this also decides whether $|g\rangle$ or $|e\rangle$ is the
   branch that swaps.

Total gate time $T=\sum_j x_j/G$ with $G=\sqrt2\,g$.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import controlled_swap as cs

%load_ext autoreload
%autoreload 2
np.set_printoptions(precision=5, suppress=True, linewidth=140)

## 1. Verify the identities (machine precision, no approximation)

In [ ]:
print("SWAP == exp(i pi n_-)                    max abs err :",
      cs.check_swap_is_dark_parity(Nc=20, Nmax=16))
print("a_+ exactly decoupled by the double drive max abs err :",
      cs.check_supermode_reduction())
print("exp(i pi n_- P_ctrl) == controlled-SWAP   fidelity    :",
      cs.ideal_conditional_parity_fidelity(Nc=20, ncut=8))

## 2. How much Fock range do we actually have to control?

$n_-$ runs up to $n_1+n_2$. **Cavities holding Fock 0..8 means the supermode sees 0..16**,
so the $e$ branch needs blocks up to $m=17$.

This is not an artefact of the supermode picture — the BS–conditional-parity–BS sandwich
needs the identical range, because after a 50:50 beamsplitter one cavity can hold every
photon. An 8-level conditional-parity gate covers only *total* $N\le7$.

In [ ]:
NMAX = 17     # blocks m = 0..17  ->  e-branch covers cavity Fock 0..16 = n1+n2
try:
    d = np.load("cswap_ge_fock8.npz")
    p, M, THETA = d["params"], int(d["M"]), float(d["theta"])
    print(f"loaded stored sequence: M = {M} segments, theta = {THETA/np.pi:+.3f} pi")
except FileNotFoundError:
    M, THETA = 40, cs.THETA_GE
    t0 = time.time()
    _, p = cs.minimize_area(NMAX, M, use_detuning=False, theta=THETA,
                            seed=23, verbose=True)
    print(f"optimised in {time.time()-t0:.0f}s")

USE_DET = (len(p) == 3 * M)
pt = cs.ParityTrain(NMAX, M, theta=THETA, use_detuning=USE_DET)
area = p[:M].sum()
print(f"\nbranch fidelity   = {pt.fidelity(p):.13f}")
print(f"total pulse area  = {area:.4f}   ->  T_gate = {area:.2f} / G,  G = sqrt(2) g")
for g_MHz in (0.5, 1.0, 2.0, 3.0):
    print(f"    g/2pi = {g_MHz:4.1f} MHz  ->  T_gate = "
          f"{area/(np.sqrt(2)*2*np.pi*g_MHz):6.2f} us")

### The pulse table

`area` is $x_j=G t_j$ (so $t_j=x_j/G$), `phase` is $\phi_j$, `Delta/G` is $r_j$.
Feed these straight into your existing `parameter_oc` "bs" segments, with the
beamsplitter acting between the transmon and the antisymmetric supermode.

In [ ]:
x, phi = p[:M], p[M:2 * M]
r = p[2 * M:3 * M] if USE_DET else np.zeros(M)
print(f"{'j':>3} {'area x_j = G t_j':>17} {'phase phi_j [rad]':>19} {'Delta_j / G':>13}")
for j in range(M):
    print(f"{j+1:3d} {x[j]:17.6f} {phi[j]:19.6f} {r[j]:13.6f}")
print(f"{'':3} {area:17.6f}   <- total area")

In [ ]:
u = pt.amplitudes(p)
u = u * np.exp(-1j * np.angle(u[0]))
n = np.arange(NMAX + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(n, np.unwrap(np.angle(u)) / np.pi, 'o-', label='achieved (g branch)')
ax[0].plot(n, THETA * n / np.pi, 'k--', lw=1, label=r'target $\theta n$')
ax[0].set_xlabel(r'$n_-$'); ax[0].set_ylabel(r'phase / $\pi$'); ax[0].legend()
ax[0].set_title('conditional phase profile')
ax[1].semilogy(n, np.maximum(1 - np.abs(u), 1e-17), 'o-')
ax[1].set_xlabel(r'$n_-$'); ax[1].set_ylabel(r'$1-|\langle n,g|U|n,g\rangle|$')
ax[1].set_title('block non-diagonality (= g-e leakage)')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Direct check of the mechanism

Every $2\times2$ block should come out **diagonal** (no g–e mixing), with
$\arg U^{gg}_m=\theta m$ and $\arg U^{ee}_m=-\arg U^{gg}_m$ (for a resonant train,
where $\gamma=0$), so the two control branches differ by exactly $\pi$ per photon.

In [ ]:
from controlled_swap import _seg2
sqn = np.sqrt(np.arange(1, NMAX + 1, dtype=float))
tot = np.broadcast_to(np.eye(2, dtype=complex), (NMAX, 2, 2)).copy()
for j in range(M):
    tot = _seg2(x[j], phi[j], r[j], sqn) @ tot
print(f"{'m':>3} {'|U_gg|':>13} {'|U_ge| (mixing)':>17} {'arg U_gg/pi':>13}"
      f" {'arg U_ee/pi':>13} {'rel/pi':>9}")
for m in range(1, NMAX + 1):
    U = tot[m - 1]
    agg, aee = np.angle(U[0, 0]) / np.pi, np.angle(U[1, 1]) / np.pi
    rel = np.angle(U[1, 1] / U[0, 0]) / np.pi
    print(f"{m:3d} {abs(U[0,0]):13.10f} {abs(U[0,1]):17.2e} {agg:13.5f}"
          f" {aee:13.5f} {rel:9.5f}")
print("\nrelative phase should advance by exactly pi (i.e. +-1 above) per photon")

In [ ]:
# (re-saving is a no-op if you loaded the stored sequence above)
np.savez("cswap_ge_fock8.npz", params=p, M=M, theta=THETA, nmax=NMAX,
         area=area, note="ge-only transmon-controlled SWAP; simultaneous "
         "equal-amplitude pi-out-of-phase double ge drive; finish with one "
         "unconditional 50:50 cavity-cavity BS exp(+i pi n_-/2)")
print("saved cswap_ge_fock8.npz")

## 3. End-to-end check in the full cavity ⊗ cavity ⊗ transmon space

No supermode assumption is used here: the real two-cavity + transmon Hamiltonian is built
for every segment and propagated. The score is the projected gate fidelity against the
ideal controlled-SWAP, allowing only the corrections that are genuinely free afterwards —
a virtual Z on each control state, and the one unconditional beamsplitter
$e^{-i\lambda n_-}e^{-i\mu N}$ that the construction calls for anyway. (Same `gauge_ops`
convention as `parameter_oc`.)

In [ ]:
for ncut in (4, 6, 8):
    t0 = time.time()
    F = cs.full_cswap_fidelity(p, M, use_detuning=USE_DET, Nc=2 * 9,
                               ncut=ncut, ctrl=(0, 1))
    print(f"cavities 0..{ncut} (total N <= {2*ncut:2d}):  F_cSWAP = {F:.10f}"
          f"   ({time.time()-t0:.0f}s)", flush=True)

### What happens if the sequence is designed for too small a Fock range

A sequence solved only to $n\le7$ is *exact* whenever the **total** photon number stays
$\le6$, and degrades badly once the cavities can hold more.

In [ ]:
pt_s = cs.ParityTrain(7, 14, theta=cs.THETA_GE, use_detuning=False)
Fs, ps = pt_s.solve(n_starts=40, seed=5)
print(f"short sequence (blocks to m=7, M=14): branch F = {Fs:.12f}, "
      f"area = {ps[:14].sum():.2f}\n")
for ncut in (2, 3, 5, 8):
    F = cs.full_cswap_fidelity(ps, 14, Nc=20, ncut=ncut, ctrl=(0, 1))
    print(f"  cavities 0..{ncut} (total N <= {2*ncut:2d}):  F_cSWAP = {F:.8f}")

## 4. Cost vs required Fock range

Gate time is roughly linear in the Fock range you insist on covering, so if your encoding
only populates low total photon number the gate gets much shorter. Weight the design by
the photon-number support of your actual code rather than by the simulation cutoff.

In [ ]:
# (a few minutes: each row runs the area-minimising search)
print(f"{'blocks to m':>12} {'cavity Fock':>12} {'M':>4} {'area G*T':>10} {'T @ g/2pi=1MHz':>16}")
for nm, Mi in ((3, 8), (5, 12), (9, 22), (13, 30)):
    out = cs.minimize_area(nm, Mi, use_detuning=False, theta=cs.THETA_GE,
                           seed=3, n_starts_first=25)
    if out is None:
        print(f"{nm:12d}  (no solution found at M={Mi})"); continue
    A = out[1][:Mi].sum()
    print(f"{nm:12d} {'0..' + str((nm-1)//2):>12} {Mi:4d} {A:10.3f}"
          f" {A/(np.sqrt(2)*2*np.pi):13.2f} us", flush=True)
print(f"{NMAX:12d} {'0..8':>12} {M:4d} {area:10.3f}"
      f" {area/(np.sqrt(2)*2*np.pi):13.2f} us   <- the sequence above")

## 5. Simple alternative — one detuned pulse, no optimal control at all

Same drive, but **detuned** and left on for $t=\pi\Delta/G^2$. That alone is
$U=P_g\,e^{-i\chi t\,n_-}+\ldots$ with $\chi=G^2/\Delta$: a controlled-SWAP with a single
calibration number and no pulse table. The price is speed — leakage $\sim4nG^2/\Delta^2$
forces $\Delta\gg G\sqrt{n_{\max}}$, which makes the gate an order of magnitude longer.

In [ ]:
print("  Delta/G    branch F        G*T")
for rr in (20, 40, 80, 160, 320, 640):
    Fd, GT = cs.dispersive_single_pulse(rr, NMAX)
    print(f"  {rr:6d}   {Fd:.9f}   {GT:8.1f}")
print(f"\noptimised sequence above:  F = {pt.fidelity(p):.9f}   G*T = {area:.1f}")